In [ ]:
from __future__ import annotations
!pip install ultralytics

import json
import math
import os
import random
import shutil
import urllib.request
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from pycocotools.coco import COCO
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import ssl                                                            # <-- Yeh add karein
ssl._create_default_https_context = ssl._create_unverified_context    # <-- Yeh add karein

# ============================================================
# 1. CONFIGURATION
# ============================================================
# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

COCO_ROOT = Path("/content/coco")
COCO_IMAGE_DIR = COCO_ROOT / "val2017"
COCO_ANNOTATION_FILE = COCO_ROOT / "annotations" / "instances_val2017.json"
OUTPUT_DIR = Path("/content/rce_coco_outputs")

AUTO_DOWNLOAD_COCO = True
MODEL_NAME = "yolov5x6"
MODEL_INPUT_SIZE = 1280
TARGET_RESULTS = 10                  # Change to any value from 5 to 10.
MAX_CANDIDATES_PER_PAIR = 100
K_CONCEPTS = 8
SOFTMAX_TEMPERATURE = 0.05
ACTIVATION_PERCENTILE = 85
BLUR_RADIUS = 15
INITIAL_DETECTION_CONF = 0.01
INTERVENTION_DETECTION_CONF = 0.01
MATCH_IOU_THRESHOLD = 0.0
SHOW_TOP_CONCEPTS = 6
SAVE_PDF = True
SAVE_COUNTERFACTUALS = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class PairSpec:
    object_a: str
    object_b: str
    max_gap_fraction: float = 0.35
    max_union_fraction: float = 0.92
    max_examples: int = 1



# سب سے عام اور آسانی سے ملنے والے COCO جوڑے
PAIR_SPECS: List[PairSpec] = [
    PairSpec("person", "laptop", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
    PairSpec("dog", "frisbee", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
    PairSpec("person", "horse", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
    PairSpec("person", "motorcycle", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
]


# ============================================================
# 2. COCO DOWNLOAD / PATH VALIDATION
# ============================================================

def _download_with_progress(url: str, destination: Path) -> None:
    import ssl
    import urllib.request
    import shutil

    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading: {url}")

    # SSL certificate error ko completely bypass karne ke liye custom context
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Bina SSL verify kiye securely download karna
    with urllib.request.urlopen(url, context=ctx) as response, open(destination, 'wb') as out_file:
        shutil.copyfileobj(response, out_file)


def _extract_zip(zip_path: Path, destination: Path) -> None:
    print(f"Extracting: {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)


def ensure_coco_files(auto_download: bool = AUTO_DOWNLOAD_COCO) -> None:
    if COCO_IMAGE_DIR.exists() and COCO_ANNOTATION_FILE.exists():
        print("COCO 2017 validation images and annotations are available.")
        return

    if not auto_download:
        raise FileNotFoundError(
            "COCO files were not found.\n"
            f"Expected images: {COCO_IMAGE_DIR}\n"
            f"Expected annotations: {COCO_ANNOTATION_FILE}\n\n"
            "Upload/mount the dataset at these paths, or set "
            "AUTO_DOWNLOAD_COCO = True and rerun."
        )

    COCO_ROOT.mkdir(parents=True, exist_ok=True)
    image_zip = COCO_ROOT / "val2017.zip"
    annotation_zip = COCO_ROOT / "annotations_trainval2017.zip"

    if not image_zip.exists():
        _download_with_progress(
            "https://images.cocodataset.org/zips/val2017.zip", image_zip
        )
    if not annotation_zip.exists():
        _download_with_progress(
            "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
            annotation_zip,
        )

    if not COCO_IMAGE_DIR.exists():
        _extract_zip(image_zip, COCO_ROOT)
    if not COCO_ANNOTATION_FILE.exists():
        _extract_zip(annotation_zip, COCO_ROOT)

    if not (COCO_IMAGE_DIR.exists() and COCO_ANNOTATION_FILE.exists()):
        raise RuntimeError("COCO download/extraction did not create the expected files.")


# ============================================================
# 3. GEOMETRY AND IMAGE HELPERS
# ============================================================

def xywh_to_xyxy(box_xywh: Sequence[float]) -> np.ndarray:
    x, y, w, h = map(float, box_xywh)
    return np.array([x, y, x + w, y + h], dtype=np.float32)


def box_area(box: np.ndarray) -> float:
    return float(max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1]))


def union_box(box_a: np.ndarray, box_b: np.ndarray) -> np.ndarray:
    return np.array(
        [
            min(box_a[0], box_b[0]),
            min(box_a[1], box_b[1]),
            max(box_a[2], box_b[2]),
            max(box_a[3], box_b[3]),
        ],
        dtype=np.float32,
    )


def calculate_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = box_area(box_a) + box_area(box_b) - intersection
    return intersection / (union + 1e-8)


def normalized_box_gap(
    box_a: np.ndarray, box_b: np.ndarray, width: int, height: int
) -> float:
    dx = max(float(box_a[0] - box_b[2]), float(box_b[0] - box_a[2]), 0.0)
    dy = max(float(box_a[1] - box_b[3]), float(box_b[1] - box_a[3]), 0.0)
    return math.hypot(dx, dy) / (math.hypot(width, height) + 1e-8)


def normalized_center_distance(
    box_a: np.ndarray, box_b: np.ndarray, width: int, height: int
) -> float:
    center_a = np.array(
        [(box_a[0] + box_a[2]) / 2.0, (box_a[1] + box_a[3]) / 2.0]
    )
    center_b = np.array(
        [(box_b[0] + box_b[2]) / 2.0, (box_b[1] + box_b[3]) / 2.0]
    )
    return float(np.linalg.norm(center_a - center_b)) / (
        math.hypot(width, height) + 1e-8
    )


def box_mask(box: np.ndarray, size: Tuple[int, int]) -> np.ndarray:
    width, height = size
    mask = np.zeros((height, width), dtype=np.uint8)
    x1, y1, x2, y2 = np.round(box).astype(int)
    x1, x2 = np.clip([x1, x2], 0, width)
    y1, y2 = np.clip([y1, y2], 0, height)
    if x2 > x1 and y2 > y1:
        mask[y1:y2, x1:x2] = 1
    return mask


def normalize_map(values: np.ndarray, support_mask: Optional[np.ndarray] = None) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if support_mask is None:
        support = np.ones(values.shape, dtype=bool)
    else:
        support = support_mask.astype(bool)
    result = np.zeros_like(values, dtype=np.float32)
    valid = values[support]
    if valid.size == 0:
        return result
    lo, hi = float(valid.min()), float(valid.max())
    if hi - lo <= 1e-8:
        return result
    result[support] = (values[support] - lo) / (hi - lo)
    return result


@dataclass
class LetterboxMeta:
    original_size: Tuple[int, int]      # width, height
    input_size: Tuple[int, int]         # width, height
    resized_size: Tuple[int, int]       # width, height
    ratio: float
    left: int
    top: int


def letterbox_rgb(
    image_rgb: np.ndarray,
    new_shape: Tuple[int, int] = (1280, 1280),
    color: Tuple[int, int, int] = (114, 114, 114),
) -> Tuple[np.ndarray, LetterboxMeta]:
    original_h, original_w = image_rgb.shape[:2]
    target_h, target_w = new_shape
    ratio = min(target_h / original_h, target_w / original_w)

    resized_w = int(round(original_w * ratio))
    resized_h = int(round(original_h * ratio))
    resized = cv2.resize(
        image_rgb, (resized_w, resized_h), interpolation=cv2.INTER_LINEAR
    )

    pad_w = target_w - resized_w
    pad_h = target_h - resized_h
    left = int(round(pad_w / 2.0 - 0.1))
    right = int(round(pad_w / 2.0 + 0.1))
    top = int(round(pad_h / 2.0 - 0.1))
    bottom = int(round(pad_h / 2.0 + 0.1))

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color
    )
    meta = LetterboxMeta(
        original_size=(original_w, original_h),
        input_size=(padded.shape[1], padded.shape[0]),
        resized_size=(resized_w, resized_h),
        ratio=ratio,
        left=left,
        top=top,
    )
    return padded, meta


def map_box_to_feature(
    box_original: np.ndarray, meta: LetterboxMeta, feature_hw: Tuple[int, int]
) -> Tuple[int, int, int, int]:
    feature_h, feature_w = feature_hw
    input_w, input_h = meta.input_size
    transformed = np.array(
        [
            box_original[0] * meta.ratio + meta.left,
            box_original[1] * meta.ratio + meta.top,
            box_original[2] * meta.ratio + meta.left,
            box_original[3] * meta.ratio + meta.top,
        ],
        dtype=np.float32,
    )
    x1 = int(np.floor(transformed[0] / input_w * feature_w))
    y1 = int(np.floor(transformed[1] / input_h * feature_h))
    x2 = int(np.ceil(transformed[2] / input_w * feature_w))
    y2 = int(np.ceil(transformed[3] / input_h * feature_h))
    x1, x2 = np.clip([x1, x2], 0, feature_w)
    y1, y2 = np.clip([y1, y2], 0, feature_h)
    return int(x1), int(y1), int(x2), int(y2)


def feature_map_to_original(
    feature_map: np.ndarray, meta: LetterboxMeta
) -> np.ndarray:
    input_w, input_h = meta.input_size
    resized_w, resized_h = meta.resized_size
    original_w, original_h = meta.original_size

    upsampled = cv2.resize(
        feature_map.astype(np.float32), (input_w, input_h), interpolation=cv2.INTER_LINEAR
    )
    crop = upsampled[
        meta.top : meta.top + resized_h,
        meta.left : meta.left + resized_w,
    ]
    if crop.size == 0:
        raise RuntimeError("Feature-map crop is empty; check letterbox metadata.")
    return cv2.resize(
        crop, (original_w, original_h), interpolation=cv2.INTER_LINEAR
    )


# ============================================================
# 4. YOLOv5 DETECTION AND FEATURE EXTRACTION
# ============================================================

@dataclass
class Detection:
    box_xyxy: np.ndarray
    score: float
    label_idx: int
    class_name: str


def _collect_4d_tensors(value: Any, output: List[torch.Tensor]) -> None:
    if torch.is_tensor(value) and value.ndim == 4:
        output.append(value)
    elif isinstance(value, (list, tuple)):
        for item in value:
            _collect_4d_tensors(item, output)
    elif isinstance(value, dict):
        for item in value.values():
            _collect_4d_tensors(item, output)


class YOLOv5Processor:
    """YOLOv5 detector plus a high-resolution neck feature capture."""

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        device: str = DEVICE,
        input_size: int = MODEL_INPUT_SIZE,
    ) -> None:
        self.device = device
        self.input_size = int(input_size)
        print(f"Loading {model_name} on {device}...")
        self.detector = torch.hub.load(
            "ultralytics/yolov5",
            model_name,
            pretrained=True,
            trust_repo=True,
        )
        self.detector = self.detector.to(device).eval()
        self.detector.max_det = 300

        self.backend = self.detector.model
        try:
            self.detect_layer = self.backend.model.model[-1]
        except Exception as exc:
            raise RuntimeError(
                "Could not resolve the YOLOv5 Detect layer. "
                "The torch-hub model structure may have changed."
            ) from exc
        self._captured_feature: Optional[torch.Tensor] = None

    def _capture_detect_inputs(self, module: torch.nn.Module, inputs: Tuple[Any, ...]) -> None:
        tensors: List[torch.Tensor] = []
        _collect_4d_tensors(inputs, tensors)
        if not tensors:
            self._captured_feature = None
            return
        # Detect receives multi-scale neck maps. Use the highest spatial resolution.
        selected = max(tensors, key=lambda tensor: tensor.shape[-2] * tensor.shape[-1])
        self._captured_feature = selected.detach().clone()

    def detect(
        self,
        image: Image.Image,
        conf_threshold: float = INITIAL_DETECTION_CONF,
        iou_threshold: float = 0.45,
    ) -> List[Detection]:
        self.detector.conf = float(conf_threshold)
        self.detector.iou = float(iou_threshold)
        with torch.no_grad():
            results = self.detector(image, size=self.input_size)

        detections: List[Detection] = []
        dataframe = results.pandas().xyxy[0]
        for _, row in dataframe.iterrows():
            label_idx = int(row["class"])
            detections.append(
                Detection(
                    box_xyxy=np.array(
                        [row.xmin, row.ymin, row.xmax, row.ymax],
                        dtype=np.float32,
                    ),
                    score=float(row.confidence),
                    label_idx=label_idx,
                    class_name=str(self.detector.names[label_idx]),
                )
            )
        return detections

    def extract_features(
        self, image: Image.Image
    ) -> Tuple[torch.Tensor, LetterboxMeta]:
        image_rgb = np.asarray(image.convert("RGB"))
        padded, meta = letterbox_rgb(
            image_rgb, new_shape=(self.input_size, self.input_size)
        )
        tensor = (
            torch.from_numpy(padded)
            .to(self.device)
            .permute(2, 0, 1)
            .contiguous()
            .unsqueeze(0)
            .float()
            / 255.0
        )
        if getattr(self.backend, "fp16", False):
            tensor = tensor.half()

        self._captured_feature = None
        handle = self.detect_layer.register_forward_pre_hook(
            self._capture_detect_inputs
        )
        try:
            with torch.no_grad():
                _ = self.backend(tensor, augment=False, visualize=False)
        finally:
            handle.remove()

        if self._captured_feature is None:
            raise RuntimeError("YOLOv5 feature capture failed.")
        return self._captured_feature, meta


def match_detection_to_ground_truth(
    detections: Sequence[Detection],
    class_name: str,
    ground_truth_box: np.ndarray,
    minimum_iou: float = MATCH_IOU_THRESHOLD,
) -> Optional[Detection]:
    candidates: List[Tuple[float, float, Detection]] = []
    for detection in detections:
        if detection.class_name != class_name:
            continue
        iou = calculate_iou(detection.box_xyxy, ground_truth_box)
        candidates.append((iou, detection.score, detection))
    if not candidates:
        return None
    best_iou, _, best_detection = max(candidates, key=lambda item: (item[0], item[1]))
    return best_detection if best_iou >= minimum_iou else None


# ============================================================
# 5. COCO PAIR SELECTION
# ============================================================

@dataclass
class GroundTruthPair:
    image_id: int
    file_name: str
    width: int
    height: int
    object_a: str
    object_b: str
    box_a: np.ndarray
    box_b: np.ndarray
    quality_score: float


class COCOPairSelector:
    def __init__(self, annotation_file: Path, image_dir: Path) -> None:
        self.coco = COCO(str(annotation_file))
        self.image_dir = Path(image_dir)
        categories = self.coco.loadCats(self.coco.getCatIds())
        self.category_id = {category["name"]: category["id"] for category in categories}

    def _annotations_for(
        self, image_id: int, category_name: str
    ) -> List[Dict[str, Any]]:
        category_id = self.category_id[category_name]
        annotation_ids = self.coco.getAnnIds(
            imgIds=[image_id], catIds=[category_id], iscrowd=False
        )
        annotations = self.coco.loadAnns(annotation_ids)
        return [
            annotation
            for annotation in annotations
            if float(annotation.get("area", 0.0)) > 1.0
        ]

    def candidate_pairs(
        self, spec: PairSpec, limit: int = MAX_CANDIDATES_PER_PAIR
    ) -> List[GroundTruthPair]:
        if spec.object_a not in self.category_id or spec.object_b not in self.category_id:
            return []

        image_ids = self.coco.getImgIds(
            catIds=[
                self.category_id[spec.object_a],
                self.category_id[spec.object_b],
            ]
        )
        scored: List[GroundTruthPair] = []

        for image_id in image_ids:
            image_info = self.coco.loadImgs([image_id])[0]
            width, height = int(image_info["width"]), int(image_info["height"])
            anns_a = self._annotations_for(image_id, spec.object_a)
            anns_b = self._annotations_for(image_id, spec.object_b)
            if not anns_a or not anns_b:
                continue

            best: Optional[GroundTruthPair] = None
            for annotation_a in anns_a:
                box_a = xywh_to_xyxy(annotation_a["bbox"])
                for annotation_b in anns_b:
                    box_b = xywh_to_xyxy(annotation_b["bbox"])
                    gap = normalized_box_gap(box_a, box_b, width, height)
                    pair_union = union_box(box_a, box_b)
                    union_fraction = box_area(pair_union) / float(width * height)
                    if gap > spec.max_gap_fraction:
                        continue
                    if union_fraction > spec.max_union_fraction:
                        continue

                    center_distance = normalized_center_distance(
                        box_a, box_b, width, height
                    )
                    # Lower is better. The score favors spatially compact,
                    # non-trivial object pairs without claiming a relation label.
                    quality = (
                        0.50 * gap
                        + 0.30 * center_distance
                        + 0.20 * union_fraction
                    )
                    candidate = GroundTruthPair(
                        image_id=int(image_id),
                        file_name=str(image_info["file_name"]),
                        width=width,
                        height=height,
                        object_a=spec.object_a,
                        object_b=spec.object_b,
                        box_a=box_a,
                        box_b=box_b,
                        quality_score=float(quality),
                    )
                    if best is None or candidate.quality_score < best.quality_score:
                        best = candidate

            if best is not None:
                scored.append(best)

        scored.sort(key=lambda item: item.quality_score)
        return scored[:limit]

    def image_path(self, pair: GroundTruthPair) -> Path:
        return self.image_dir / pair.file_name


# ============================================================
# 6. CONCEPT DISCOVERY AND INTERVENTIONAL SCORING
# ============================================================

@dataclass
class ConceptResult:
    concept_index: int
    effect_score: float
    original_score_a: float
    original_score_b: float
    post_score_a: float
    post_score_b: float
    mask_fraction: float
    auto_region_label: str
    activation_map: np.ndarray
    intervention_mask: np.ndarray


class KMeansConceptExplainer:
    """K-Means over all feature vectors inside the selected union region."""

    def __init__(
        self,
        number_of_concepts: int = K_CONCEPTS,
        temperature: float = SOFTMAX_TEMPERATURE,
        seed: int = SEED,
    ) -> None:
        self.number_of_concepts = int(number_of_concepts)
        self.temperature = float(temperature)
        self.seed = int(seed)

    def discover(
        self,
        feature_tensor: torch.Tensor,
        relational_box: np.ndarray,
        meta: LetterboxMeta,
    ) -> torch.Tensor:
        if feature_tensor.ndim != 4 or feature_tensor.shape[0] != 1:
            raise ValueError(
                f"Expected a [1,C,H,W] feature tensor, got {feature_tensor.shape}."
            )

        _, channels, feature_h, feature_w = feature_tensor.shape
        x1, y1, x2, y2 = map_box_to_feature(
            relational_box, meta, (feature_h, feature_w)
        )
        if x2 <= x1 or y2 <= y1:
            raise RuntimeError("The relational box maps to an empty feature region.")

        feature_grid = feature_tensor[0].permute(1, 2, 0).contiguous()
        region_vectors = feature_grid[y1:y2, x1:x2].reshape(-1, channels)
        region_vectors = region_vectors[
            torch.isfinite(region_vectors).all(dim=1)
        ]
        if region_vectors.shape[0] < 2:
            raise RuntimeError("Too few valid feature vectors for concept discovery.")

        k_final = min(self.number_of_concepts, int(region_vectors.shape[0]))
        kmeans = KMeans(
            n_clusters=k_final,
            random_state=self.seed,
            n_init=10,
        )
        centers_numpy = kmeans.fit(region_vectors.cpu().numpy()).cluster_centers_
        centers = torch.as_tensor(
            centers_numpy,
            dtype=feature_tensor.dtype,
            device=feature_tensor.device,
        )

        all_vectors = feature_grid.reshape(-1, channels)
        distances = torch.cdist(all_vectors.float(), centers.float())
        similarity = F.softmax(-distances / self.temperature, dim=1)
        activation_maps = similarity.T.reshape(k_final, feature_h, feature_w)
        return activation_maps.unsqueeze(0)


def automatic_region_label(
    activation_map: np.ndarray,
    detection_a: Detection,
    detection_b: Detection,
    pair_union: np.ndarray,
) -> str:
    support = box_mask(pair_union, (activation_map.shape[1], activation_map.shape[0]))
    normalized = normalize_map(activation_map, support)
    total = float((normalized * support).sum()) + 1e-8

    mass_a = float((normalized * box_mask(detection_a.box_xyxy, (activation_map.shape[1], activation_map.shape[0]))).sum()) / total
    mass_b = float((normalized * box_mask(detection_b.box_xyxy, (activation_map.shape[1], activation_map.shape[0]))).sum()) / total
    context_mass = max(0.0, 1.0 - min(1.0, mass_a + mass_b))

    if mass_a >= 0.25 and mass_b >= 0.25:
        return "shared pair region"
    if mass_a >= max(0.20, mass_b):
        return f"{detection_a.class_name}-dominant region"
    if mass_b >= max(0.20, mass_a):
        return f"{detection_b.class_name}-dominant region"
    if context_mass >= 0.45:
        return "shared context"
    return "mixed relational region"


class RelationalInterventionExplainer:
    def __init__(
        self,
        processor: YOLOv5Processor,
        percentile: float = ACTIVATION_PERCENTILE,
        blur_radius: float = BLUR_RADIUS,
        matching_iou: float = MATCH_IOU_THRESHOLD,
    ) -> None:
        self.processor = processor
        self.percentile = float(percentile)
        self.blur_radius = float(blur_radius)
        self.matching_iou = float(matching_iou)

    def _matched_post_score(
        self,
        detections: Sequence[Detection],
        original_detection: Detection,
    ) -> float:
        candidates: List[Tuple[float, float]] = []
        for detection in detections:
            if detection.class_name != original_detection.class_name:
                continue
            iou = calculate_iou(
                original_detection.box_xyxy, detection.box_xyxy
            )
            if iou >= self.matching_iou:
                candidates.append((iou, detection.score))
        if not candidates:
            return 0.0
        return max(candidates, key=lambda item: (item[0], item[1]))[1]

    def evaluate(
        self,
        image: Image.Image,
        activation_maps: torch.Tensor,
        meta: LetterboxMeta,
        detection_a: Detection,
        detection_b: Detection,
        pair_union: np.ndarray,
        counterfactual_dir: Optional[Path] = None,
    ) -> List[ConceptResult]:
        image_rgb = np.asarray(image.convert("RGB"))
        width, height = image.size
        union_support = box_mask(pair_union, image.size)
        blurred = np.asarray(
            image.filter(ImageFilter.GaussianBlur(radius=self.blur_radius))
        )
        original_joint = detection_a.score * detection_b.score

        results: List[ConceptResult] = []
        for concept_index in range(activation_maps.shape[1]):
            feature_map = (
                activation_maps[0, concept_index].detach().cpu().numpy()
            )
            original_map = feature_map_to_original(feature_map, meta)
            normalized = normalize_map(original_map, union_support)

            values = normalized[union_support.astype(bool)]
            if values.size == 0 or float(values.max()) <= 0.0:
                threshold = 1.0
            else:
                threshold = float(np.percentile(values, self.percentile))

            intervention_mask = (
                (normalized >= threshold) & (union_support > 0)
            ).astype(np.uint8)
            mask_fraction = float(intervention_mask.sum()) / (
                float(union_support.sum()) + 1e-8
            )

            counterfactual = np.where(
                intervention_mask[..., None] > 0,
                blurred,
                image_rgb,
            ).astype(np.uint8)
            counterfactual_image = Image.fromarray(counterfactual)

            post_detections = self.processor.detect(
                counterfactual_image,
                conf_threshold=INTERVENTION_DETECTION_CONF,
            )
            post_a = self._matched_post_score(post_detections, detection_a)
            post_b = self._matched_post_score(post_detections, detection_b)

            # ہر آبجیکٹ کے سکور میں کتنی کمی آئی؟
            drop_a = max(0.0, float(detection_a.score) - post_a)
            drop_b = max(0.0, float(detection_b.score) - post_b)

            # XAI Correlation Metric: ہم min() استعمال کریں گے تاکہ صرف وہ حصہ
            # ٹاپ سکور حاصل کرے جو دونوں آبجیکٹس (Correlated) کو متاثر کرے۔
            effect = float(min(drop_a, drop_b))

            if counterfactual_dir is not None and SAVE_COUNTERFACTUALS:
                counterfactual_dir.mkdir(parents=True, exist_ok=True)
                counterfactual_image.save(
                    counterfactual_dir / f"concept_{concept_index + 1:02d}.png"
                )

            results.append(
                ConceptResult(
                    concept_index=concept_index,
                    effect_score=float(effect),
                    original_score_a=float(detection_a.score),
                    original_score_b=float(detection_b.score),
                    post_score_a=float(post_a),
                    post_score_b=float(post_b),
                    mask_fraction=float(mask_fraction),
                    auto_region_label=automatic_region_label(
                        normalized,
                        detection_a,
                        detection_b,
                        pair_union,
                    ),
                    activation_map=normalized,
                    intervention_mask=intervention_mask,
                )
            )
        return results


# ============================================================
# 7. PAPER-READY VISUALIZATION
# ============================================================

def draw_labelled_box(
    image_rgb: np.ndarray,
    detection: Detection,
    color: Tuple[int, int, int],
) -> None:
    x1, y1, x2, y2 = np.round(detection.box_xyxy).astype(int)
    cv2.rectangle(image_rgb, (x1, y1), (x2, y2), color, 3)
    label = f"{detection.class_name} {detection.score:.2f}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness = 0.65, 2
    (text_w, text_h), baseline = cv2.getTextSize(
        label, font, scale, thickness
    )
    label_top = max(0, y1 - text_h - baseline - 8)
    cv2.rectangle(
        image_rgb,
        (x1, label_top),
        (x1 + text_w + 8, label_top + text_h + baseline + 8),
        color,
        -1,
    )
    cv2.putText(
        image_rgb,
        label,
        (x1 + 4, label_top + text_h + 3),
        font,
        scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA,
    )


def make_union_overlay(
    image_rgb: np.ndarray, pair_union: np.ndarray
) -> np.ndarray:
    result = image_rgb.copy()
    overlay = result.copy()
    x1, y1, x2, y2 = np.round(pair_union).astype(int)
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (255, 0, 255), -1)
    result = cv2.addWeighted(overlay, 0.32, result, 0.68, 0.0)
    cv2.rectangle(result, (x1, y1), (x2, y2), (255, 0, 255), 4)
    return result


def combined_top_concept_heatmap(
    concept_results: Sequence[ConceptResult],
    pair_union: np.ndarray,
    image_size: Tuple[int, int],
    top_n: int = 3,
) -> np.ndarray:
    width, height = image_size
    support = box_mask(pair_union, image_size)
    ranked = sorted(
        concept_results, key=lambda result: result.effect_score, reverse=True
    )
    selected = ranked[: max(1, min(top_n, len(ranked)))]

    positive_weights = np.array(
        [max(0.0, result.effect_score) for result in selected],
        dtype=np.float32,
    )
    if float(positive_weights.sum()) <= 1e-8:
        positive_weights = np.ones(len(selected), dtype=np.float32)
    positive_weights /= positive_weights.sum()

    heatmap = np.zeros((height, width), dtype=np.float32)
    for weight, result in zip(positive_weights, selected):
        heatmap += weight * result.activation_map

    heatmap *= support
    heatmap = normalize_map(heatmap, support)
    sigma = max(2.0, min(width, height) / 120.0)
    heatmap = cv2.GaussianBlur(heatmap, (0, 0), sigmaX=sigma, sigmaY=sigma)
    return normalize_map(heatmap, support)


def overlay_heatmap(image_rgb: np.ndarray, heatmap: np.ndarray) -> np.ndarray:
    colored_bgr = cv2.applyColorMap(
        np.uint8(np.clip(heatmap, 0.0, 1.0) * 255),
        cv2.COLORMAP_JET,
    )
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)
    nonzero = heatmap[heatmap > 0]
    if nonzero.size == 0:
        return image_rgb.copy()

    threshold = float(np.percentile(nonzero, 60))
    alpha = np.clip((heatmap - threshold) / (1.0 - threshold + 1e-8), 0.0, 1.0)
    alpha = (0.78 * alpha)[..., None]
    darkened = (image_rgb.astype(np.float32) * 0.48)
    composed = darkened * (1.0 - alpha) + colored_rgb.astype(np.float32) * alpha
    return np.uint8(np.clip(composed, 0, 255))


def create_dashboard(
    image: Image.Image,
    image_id: int,
    detection_a: Detection,
    detection_b: Detection,
    pair_union: np.ndarray,
    concept_results: Sequence[ConceptResult],
    output_stem: Path,
) -> Path:
    image_rgb = np.asarray(image.convert("RGB"))
    boxed = image_rgb.copy()
    draw_labelled_box(boxed, detection_a, (0, 255, 255))
    draw_labelled_box(boxed, detection_b, (255, 255, 0))
    union_view = make_union_overlay(image_rgb, pair_union)

    heatmap = combined_top_concept_heatmap(
        concept_results, pair_union, image.size, top_n=3
    )
    heatmap_view = overlay_heatmap(image_rgb, heatmap)

    ranked = sorted(
        concept_results, key=lambda result: result.effect_score, reverse=True
    )
    displayed = ranked[: min(SHOW_TOP_CONCEPTS, len(ranked))]
    labels = [
        f"C{result.concept_index + 1}: {result.auto_region_label}"
        for result in displayed
    ]
    scores = [result.effect_score for result in displayed]

    fig, axes = plt.subplots(2, 2, figsize=(13.2, 10.5))
    fig.suptitle(
        f"Qualitative RCE Example: {detection_a.class_name} + "
        f"{detection_b.class_name}  |  COCO image {image_id}",
        fontsize=16,
        fontweight="bold",
    )

    axes[0, 0].imshow(boxed)
    axes[0, 0].set_title("1. Interacting Objects Detected", fontweight="bold")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(union_view)
    axes[0, 1].set_title("2. Relational Analysis Region", fontweight="bold")
    axes[0, 1].axis("off")

    axes[1, 0].imshow(heatmap_view)
    axes[1, 0].set_title("3. Top-Ranked RCE Concept Overlay", fontweight="bold")
    axes[1, 0].axis("off")

    y_positions = np.arange(len(labels))
    bar_colors = ["coral" if value >= 0 else "lightgray" for value in scores]
    axes[1, 1].barh(
        y_positions,
        scores,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.8,
    )
    axes[1, 1].set_yticks(y_positions)
    axes[1, 1].set_yticklabels(labels, fontsize=9)
    axes[1, 1].invert_yaxis()
    axes[1, 1].axvline(0.0, linewidth=1.0, color="black")
    axes[1, 1].set_xlabel(r"Concept-Effect Score $CE_k$")
    axes[1, 1].set_title("4. Interventional Concept Ranking", fontweight="bold")
    axes[1, 1].grid(axis="x", linestyle=":", alpha=0.35)

    for y_value, score in zip(y_positions, scores):
        offset = 0.008 * max(1e-3, max(abs(np.asarray(scores))))
        horizontal = score + offset if score >= 0 else score - offset
        alignment = "left" if score >= 0 else "right"
        axes[1, 1].text(
            horizontal,
            y_value,
            f"{score:.3f}",
            va="center",
            ha=alignment,
            fontsize=8,
        )

    fig.text(
        0.5,
        0.012,
        "Concept descriptions are automatically derived from activation "
        "overlap and are not semantic labels generated by RCE.",
        ha="center",
        fontsize=9,
    )
    plt.tight_layout(rect=[0.0, 0.035, 1.0, 0.95])

    png_path = output_stem.with_suffix(".png")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return png_path


def create_contact_sheet(
    dashboard_paths: Sequence[Path],
    output_path: Path,
    columns: int = 2,
    panel_width: int = 1700,
    margin: int = 30,
) -> Path:
    if not dashboard_paths:
        raise ValueError("No dashboard paths were supplied.")

    panels: List[Image.Image] = []
    for dashboard_path in dashboard_paths:
        panel = Image.open(dashboard_path).convert("RGB")
        scale = panel_width / float(panel.width)
        panel = panel.resize(
            (panel_width, int(round(panel.height * scale))),
            Image.Resampling.LANCZOS,
        )
        panels.append(panel)

    rows = math.ceil(len(panels) / columns)
    row_heights: List[int] = []
    for row in range(rows):
        row_panels = panels[row * columns : (row + 1) * columns]
        row_heights.append(max(panel.height for panel in row_panels))

    sheet_width = columns * panel_width + (columns + 1) * margin
    sheet_height = sum(row_heights) + (rows + 1) * margin
    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")

    y = margin
    for row in range(rows):
        x = margin
        row_panels = panels[row * columns : (row + 1) * columns]
        for panel in row_panels:
            sheet.paste(panel, (x, y))
            x += panel_width + margin
        y += row_heights[row] + margin

    sheet.save(output_path, dpi=(300, 300))
    return output_path


# ============================================================
# 8. END-TO-END RUNNER
# ============================================================

def analyze_ground_truth_pair(
    processor: YOLOv5Processor,
    pair: GroundTruthPair,
    selector: COCOPairSelector,
) -> Optional[Dict[str, Any]]:
    image_path = selector.image_path(pair)
    if not image_path.exists():
        return None

    image = Image.open(image_path).convert("RGB")
    img_width, img_height = image.size
    img_area = img_width * img_height

    detections = processor.detect(
        image, conf_threshold=INITIAL_DETECTION_CONF
    )

    detection_a = match_detection_to_ground_truth(
        detections, pair.object_a, pair.box_a, minimum_iou=MATCH_IOU_THRESHOLD
    )
    detection_b = match_detection_to_ground_truth(
        detections, pair.object_b, pair.box_b, minimum_iou=MATCH_IOU_THRESHOLD
    )

    if detection_a is None or detection_b is None:
        return None

    # ==============================================================
    # NAYA IZAFA: Object Size Check (صرف بڑے اور سامنے والے آبجیکٹس کے لیے)
    # ==============================================================
    area_a = box_area(detection_a.box_xyxy)
    area_b = box_area(detection_b.box_xyxy)

    # اگر کوئی بھی آبجیکٹ پوری تصویر کے 4% یا 3% سے چھوٹا ہے، تو اسے چھوڑ دیں
    if area_a < (img_area * 0.04) or area_b < (img_area * 0.03):
        print(f"\n[Skipped] Image {pair.image_id} - Objects are too small (Background).")
        return None
    # ==============================================================

    pair_union = union_box(
        detection_a.box_xyxy, detection_b.box_xyxy
    )
    features, meta = processor.extract_features(image)
    concept_explainer = KMeansConceptExplainer()
    activation_maps = concept_explainer.discover(
        features, pair_union, meta
    )

    intervention_explainer = RelationalInterventionExplainer(processor, matching_iou=MATCH_IOU_THRESHOLD)
    example_stem = (
        f"coco_{pair.image_id}_"
        f"{pair.object_a.replace(' ', '_')}_"
        f"{pair.object_b.replace(' ', '_')}"
    )
    example_dir = OUTPUT_DIR / example_stem
    example_dir.mkdir(parents=True, exist_ok=True)

    concept_results = intervention_explainer.evaluate(
        image=image,
        activation_maps=activation_maps,
        meta=meta,
        detection_a=detection_a,
        detection_b=detection_b,
        pair_union=pair_union,
        counterfactual_dir=example_dir / "counterfactuals",
    )

    dashboard_path = create_dashboard(
        image=image,
        image_id=pair.image_id,
        detection_a=detection_a,
        detection_b=detection_b,
        pair_union=pair_union,
        concept_results=concept_results,
        output_stem=example_dir / "dashboard",
    )

    rows: List[Dict[str, Any]] = []
    for result in concept_results:
        rows.append(
            {
                "image_id": pair.image_id,
                "file_name": pair.file_name,
                "object_a": pair.object_a,
                "object_b": pair.object_b,
                "concept_id": result.concept_index + 1,
                "auto_region_label": result.auto_region_label,
                "effect_score_CE_k": result.effect_score,
                "original_score_a": result.original_score_a,
                "original_score_b": result.original_score_b,
                "original_joint_score": (
                    result.original_score_a * result.original_score_b
                ),
                "post_score_a": result.post_score_a,
                "post_score_b": result.post_score_b,
                "post_joint_score": (
                    result.post_score_a * result.post_score_b
                ),
                "mask_fraction_of_union": result.mask_fraction,
            }
        )
    pd.DataFrame(rows).to_csv(
        example_dir / "concept_scores.csv", index=False
    )

    summary = {
        "image_id": pair.image_id,
        "file_name": pair.file_name,
        "object_a": pair.object_a,
        "object_b": pair.object_b,
        "detector_score_a": detection_a.score,
        "detector_score_b": detection_b.score,
        "original_joint_score": detection_a.score * detection_b.score,
        "top_effect_score": max(
            result.effect_score for result in concept_results
        ),
        "dashboard_path": str(dashboard_path),
    }
    with open(example_dir / "summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)
    return summary


def run_coco_generation() -> pd.DataFrame:
    ensure_coco_files()
    selector = COCOPairSelector(
        COCO_ANNOTATION_FILE, COCO_IMAGE_DIR
    )
    processor = YOLOv5Processor()

    summaries: List[Dict[str, Any]] = []
    dashboard_paths: List[Path] = []
    used_image_ids: set[int] = set()

    for spec in PAIR_SPECS:
        if len(summaries) >= TARGET_RESULTS:
            break

        print(f"\nSearching COCO pair: {spec.object_a} + {spec.object_b}")
        candidates = selector.candidate_pairs(spec)
        successes_for_pair = 0

        for candidate in tqdm(candidates, desc=f"{spec.object_a}+{spec.object_b}"):
            if candidate.image_id in used_image_ids:
                continue
            try:
                summary = analyze_ground_truth_pair(
                    processor, candidate, selector
                )
            except Exception as error:
                print(
                    f"Skipped COCO image {candidate.image_id}: "
                    f"{type(error).__name__}: {error}"
                )
                continue

            if summary is None:
                continue

            summaries.append(summary)
            dashboard_paths.append(Path(summary["dashboard_path"]))
            used_image_ids.add(candidate.image_id)
            successes_for_pair += 1
            print(
                f"Saved result {len(summaries)}/{TARGET_RESULTS}: "
                f"COCO {candidate.image_id}, "
                f"{spec.object_a} + {spec.object_b}"
            )

            if successes_for_pair >= spec.max_examples:
                break
            if len(summaries) >= TARGET_RESULTS:
                break

    if len(summaries) < TARGET_RESULTS:
        print(
            f"\nWarning: only {len(summaries)} valid results were generated. "
            "Increase MAX_CANDIDATES_PER_PAIR, lower the initial confidence "
            "slightly, or add more PairSpec entries."
        )

    summary_frame = pd.DataFrame(summaries)
    summary_frame.to_csv(OUTPUT_DIR / "all_results_summary.csv", index=False)

    manifest = {
        "configuration": {
            "model": MODEL_NAME,
            "input_size": MODEL_INPUT_SIZE,
            "target_results": TARGET_RESULTS,
            "K": K_CONCEPTS,
            "temperature": SOFTMAX_TEMPERATURE,
            "activation_percentile": ACTIVATION_PERCENTILE,
            "blur_radius": BLUR_RADIUS,
            "initial_detection_conf": INITIAL_DETECTION_CONF,
            "intervention_detection_conf": INTERVENTION_DETECTION_CONF,
            "matching_iou_threshold": MATCH_IOU_THRESHOLD,
            "seed": SEED,
        },
        "results": summaries,
    }
    with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)

    if dashboard_paths:
        create_contact_sheet(
            dashboard_paths,
            OUTPUT_DIR / f"composite_{len(dashboard_paths)}_examples.png",
            columns=2,
        )

    print(f"\nFinished. Outputs are in: {OUTPUT_DIR}")
    return summary_frame


if __name__ == "__main__":
    results = run_coco_generation()
    print(results.to_string(index=False))